# 20 — Resolve CDISC CT for COSMoS Graph

Second step in the cosmos-graph pipeline — resolves the concept IDs carried by the structural graph (`Codelists.codelist_concept_id`, `Variables.assigned_term_concept_id`) against the NCI EVS SDTM Controlled Terminology package.

Adds semantic content (submission values, synonyms, definitions, NCI preferred terms) as separate lookup sheets. The `Variables` and `Codelists` columns keep their authored C-codes unchanged — consumers join on ID.

**Input**
- `interim/COSMoS_Graph.xlsx` — output of notebook 10
- `cosmos-bc-dss/downloads/SDTM_Terminology.txt` — NCI EVS SDTM CT package (tab-separated, 8 columns)

**Output**
- `interim/COSMoS_Graph_CT.xlsx` with sheets: `ReadMe`, `Codelists`, `CodelistTerms`, `AssignedTerms`, `Unresolved`, `Anomalies`

**Out of scope** — NCIt enrichment beyond what the SDTM CT package carries (UMLS mappings, hierarchy, cross-ontology identity) is Step 3. Inline `value_list` resolution and `subset_codelist` parsing stay as-authored (Step 2 design decisions).

Design rationale: [`../docs/COSMoS_Graph.md`](../docs/COSMoS_Graph.md).


## 1. Imports


In [1]:
import sys
from pathlib import Path
from datetime import date

import pandas as pd

print(f'python : {sys.version.split()[0]}')
print(f'pandas : {pd.__version__}')


python : 3.11.15
pandas : 3.0.2


## 2. Configuration


In [2]:
REPO_ROOT = Path.cwd().parent.parent  # notebooks/ -> cosmos-graph/ -> repo root

DOWNLOADS = REPO_ROOT / 'cosmos-bc-dss' / 'downloads'
INTERIM   = REPO_ROOT / 'cosmos-graph' / 'interim'

GRAPH_XLSX = INTERIM / 'COSMoS_Graph.xlsx'
CT_TXT     = DOWNLOADS / 'SDTM_Terminology.txt'
OUT_XLSX   = INTERIM / 'COSMoS_Graph_CT.xlsx'

for p in (GRAPH_XLSX, CT_TXT):
    assert p.exists(), f'missing: {p}'

print(f'in  : {GRAPH_XLSX.relative_to(REPO_ROOT)}')
print(f'CT  : {CT_TXT.relative_to(REPO_ROOT)}')
print(f'out : {OUT_XLSX.relative_to(REPO_ROOT)}')


in  : cosmos-graph/interim/COSMoS_Graph.xlsx
CT  : cosmos-bc-dss/downloads/SDTM_Terminology.txt
out : cosmos-graph/interim/COSMoS_Graph_CT.xlsx


## 3. Load the structural graph

The two sheets we need to enrich: `Codelists` (referenced codelist bindings) and `Variables` (source of pinned `assigned_term_concept_id` values).


In [3]:
codelists_in  = pd.read_excel(GRAPH_XLSX, sheet_name='Codelists')
variables_in  = pd.read_excel(GRAPH_XLSX, sheet_name='Variables')

print(f'Codelists : {codelists_in.shape}')
print(f'Variables : {variables_in.shape}')

referenced_codelist_ids = set(codelists_in['codelist_concept_id'].dropna().unique())
referenced_term_ids     = set(variables_in['assigned_term_concept_id'].dropna().unique())
print(f'\ndistinct codelist C-codes referenced : {len(referenced_codelist_ids):,}')
print(f'distinct assigned-term C-codes         : {len(referenced_term_ids):,}')


Codelists : (297, 3)
Variables : (13922, 26)

distinct codelist C-codes referenced : 297
distinct assigned-term C-codes         : 1,307


## 4. Load SDTM CT package

The NCI EVS file has a single table with two row types distinguished by whether `Codelist Code` is populated:

- **Codelist-level rows** — `Code` is the codelist C-code, `Codelist Code` is empty. `Codelist Name`, `Codelist Extensible`, and `NCI Preferred Term` describe the codelist itself.
- **Term-level rows** — `Code` is a term C-code, `Codelist Code` is the parent codelist's C-code. `CDISC Submission Value` is the term's submission value; `CDISC Synonym(s)` and `CDISC Definition` describe the term.

We split the table on this structural distinction and then join against our references.


In [4]:
ct_raw = pd.read_csv(CT_TXT, sep='\t', dtype=str, keep_default_na=False, na_values=[''])
print(f'SDTM_Terminology.txt : {ct_raw.shape}')
print(f'columns              : {list(ct_raw.columns)}')

# Rename to canonical snake_case
CT_RENAME = {
    'Code':                         'concept_id',
    'Codelist Code':                'parent_codelist_concept_id',
    'Codelist Extensible (Yes/No)': 'codelist_extensible',
    'Codelist Name':                'codelist_name',
    'CDISC Submission Value':       'submission_value',
    'CDISC Synonym(s)':             'synonyms',
    'CDISC Definition':             'definition',
    'NCI Preferred Term':           'nci_preferred_term',
}
ct = ct_raw.rename(columns=CT_RENAME)

is_codelist_row = ct['parent_codelist_concept_id'].isna()
ct_codelist = ct.loc[ is_codelist_row].copy()
ct_term     = ct.loc[~is_codelist_row].copy()

print(f'\ncodelist-level rows : {len(ct_codelist):,}')
print(f'term-level rows     : {len(ct_term):,}')


SDTM_Terminology.txt : (46774, 8)
columns              : ['Code', 'Codelist Code', 'Codelist Extensible (Yes/No)', 'Codelist Name', 'CDISC Submission Value', 'CDISC Synonym(s)', 'CDISC Definition', 'NCI Preferred Term']

codelist-level rows : 1,208
term-level rows     : 45,566


## 5. Build `Codelists` sheet — codelist-level enrichment

Join each referenced codelist to its CT codelist-level row to pick up `codelist_name`, `codelist_extensible`, and `codelist_nci_preferred_term`. The input `codelist_submission_value` is preserved as authored; the CT-derived submission value is added alongside for cross-check.


In [5]:
codelist_enrich = ct_codelist[[
    'concept_id', 'submission_value', 'codelist_name',
    'codelist_extensible', 'nci_preferred_term',
]].rename(columns={
    'concept_id':         'codelist_concept_id',
    'submission_value':   'codelist_submission_value_ct',
    'nci_preferred_term': 'codelist_nci_preferred_term',
})

codelists_sheet = codelists_in.merge(codelist_enrich, on='codelist_concept_id', how='left')

# Consistency check — where CT resolved, authored submission_value should match CT submission_value
resolved = codelists_sheet['codelist_submission_value_ct'].notna()
mismatch = resolved & (
    codelists_sheet['codelist_submission_value']
    != codelists_sheet['codelist_submission_value_ct']
)

print(f'Codelists sheet: {codelists_sheet.shape}')
print(f'resolved: {resolved.sum():,}  unresolved: {(~resolved).sum():,}')
print(f'submission_value mismatches (authored vs CT): {mismatch.sum()}')
if mismatch.any():
    print(codelists_sheet.loc[mismatch, [
        'codelist_concept_id', 'codelist_submission_value', 'codelist_submission_value_ct',
    ]].to_string())
codelists_sheet.head()


Codelists sheet: (297, 7)
resolved: 295  unresolved: 2
submission_value mismatches (authored vs CT): 0


,codelist_concept_id,codelist_submission_value,variable_uses_count,codelist_submission_value_ct,codelist_name,codelist_extensible,codelist_nci_preferred_term
0,C100129,QSCAT,17,QSCAT,Category of Questionnaire,Yes,CDISC Questionnaire Category Terminology
1,C100131,ADCTN,16,ADCTN,Alzheimer's Disease Assessment Scale - Cogniti...,No,CDISC Functional Test ADAS-Cog CDISC Version T...
2,C100132,ADCTC,16,ADCTC,Alzheimer's Disease Assessment Scale - Cogniti...,No,CDISC Functional Test ADAS-Cog CDISC Version T...
3,C100133,BPRSA1TN,18,BPRSA1TN,Brief Psychiatric Rating Scale-Anchored Clinic...,No,CDISC Clinical Classification BPRS-Anchored Te...
4,C100134,BPRSA1TC,18,BPRSA1TC,Brief Psychiatric Rating Scale-Anchored Clinic...,No,CDISC Clinical Classification BPRS-Anchored Te...


## 6. Build `CodelistTerms` sheet — full permissible value sets

For every referenced codelist, expand to all its terms. Columns: codelist identity (`codelist_concept_id`, `codelist_submission_value`) joined with term identity (`term_concept_id`, `term_submission_value`, `term_synonyms`, `term_definition`, `term_nci_preferred_term`).

Grain: one row per (codelist, term). A consumer answering 'what values can this DSS bind?' joins `Variables.codelist_concept_id` → `CodelistTerms.codelist_concept_id`.


In [6]:
terms_for_referenced = ct_term[
    ct_term['parent_codelist_concept_id'].isin(referenced_codelist_ids)
].copy()

# Bring in codelist-level submission value for convenience
cl_sub = ct_codelist.set_index('concept_id')['submission_value'].to_dict()
terms_for_referenced['codelist_submission_value'] = (
    terms_for_referenced['parent_codelist_concept_id'].map(cl_sub)
)

codelist_terms_sheet = terms_for_referenced[[
    'parent_codelist_concept_id', 'codelist_submission_value',
    'concept_id', 'submission_value', 'synonyms', 'definition', 'nci_preferred_term',
]].rename(columns={
    'parent_codelist_concept_id': 'codelist_concept_id',
    'concept_id':                 'term_concept_id',
    'submission_value':           'term_submission_value',
    'synonyms':                   'term_synonyms',
    'definition':                 'term_definition',
    'nci_preferred_term':         'term_nci_preferred_term',
}).sort_values(['codelist_concept_id', 'term_submission_value']).reset_index(drop=True)

print(f'CodelistTerms sheet: {codelist_terms_sheet.shape}')
print(f'distinct codelists covered: {codelist_terms_sheet.codelist_concept_id.nunique()}')
avg_terms = codelist_terms_sheet.groupby('codelist_concept_id').size().mean()
print(f'average terms per codelist: {avg_terms:.1f}')
codelist_terms_sheet.head()


CodelistTerms sheet: (17607, 7)
distinct codelists covered: 295
average terms per codelist: 59.7


,codelist_concept_id,codelist_submission_value,term_concept_id,term_submission_value,term_synonyms,term_definition,term_nci_preferred_term
0,C100129,QSCAT,C187516,ABC,ABC01,Activities-Specific Balance Confidence Scale (...,Activities-Specific Balance Confidence Scale Q...
1,C100129,QSCAT,C122370,ACQ,ACQ01,Asthma Control Questionnaire (ACQ) (The Asthma...,Asthma Control Questionnaire
2,C100129,QSCAT,C123658,ACT,ACT01,Asthma Control Test (ACT) (Asthma Control Test...,Asthma Control Test Questionnaire
3,C100129,QSCAT,C105166,ADCS-ADL,ADL01,Alzheimer's Disease Cooperative Study-Activiti...,Alzheimer's Disease Cooperative Study Activiti...
4,C100129,QSCAT,C106888,ADCS-ADL MCI,ADL03,Alzheimer's Disease Cooperative Study-Activiti...,Alzheimer's Disease Cooperative Study-Activiti...


## 7. Build `AssignedTerms` sheet — pinned term identity (context-stable fields only)

For each distinct `assigned_term_concept_id` referenced by `Variables`, resolve the term-level identity.

**Grain**: one row per distinct `assigned_term_concept_id`.

**Columns**: only fields that are stable per C-code across codelist contexts — `term_definition`, `term_nci_preferred_term`. The context-variant `submission_value` and `synonyms` are deliberately NOT included here: the same C-code can carry different CDISC submission values in different codelists (e.g., `Length` vs `LENGTH` for TEST vs TESTCD), and synonyms often include codelist-context prefixes.

**To get context-specific submission_value / synonyms**, join `Variables` to `CodelistTerms` on both `codelist_concept_id` AND `assigned_term_concept_id` (= `term_concept_id`).


In [7]:
# Count variable uses per pinned term
term_usage = (
    variables_in['assigned_term_concept_id']
    .dropna()
    .value_counts()
    .rename_axis('assigned_term_concept_id')
    .reset_index(name='used_in_variables_count')
)

# Stable-per-C-code fields only. definition and nci_preferred_term are invariant across
# codelist contexts (verified empirically — 0 within-code variance on the 2026-Q1 data).
# submission_value and synonyms vary by codelist context and belong in CodelistTerms.
assigned_enrich = (
    ct[['concept_id', 'definition', 'nci_preferred_term']]
    .rename(columns={
        'concept_id':         'assigned_term_concept_id',
        'definition':         'term_definition',
        'nci_preferred_term': 'term_nci_preferred_term',
    })
    .drop_duplicates(subset=['assigned_term_concept_id'])
)

assigned_terms_sheet = term_usage.merge(
    assigned_enrich, on='assigned_term_concept_id', how='left'
).sort_values('used_in_variables_count', ascending=False).reset_index(drop=True)

assert assigned_terms_sheet['assigned_term_concept_id'].is_unique, 'non-unique pinned term C-codes after dedup'

resolved = assigned_terms_sheet['term_definition'].notna()
print(f'AssignedTerms sheet: {assigned_terms_sheet.shape}')
print(f'resolved: {resolved.sum():,}  unresolved: {(~resolved).sum():,}')
assigned_terms_sheet.head(10)


AssignedTerms sheet: (1307, 4)
resolved: 1,305  unresolved: 2


,assigned_term_concept_id,used_in_variables_count,term_definition,term_nci_preferred_term
0,C181398,256,A measurement of the binding allergen-induced ...,Allergen-induced IgE Antibody Measurement
1,C181394,126,A measurement of the binding microbial-induced...,Microbial-induced IgG Antibody Measurement
2,C187777,106,A measurement of the binding allergen-induced ...,Allergen-induced IgG Antibody Measurement
3,C187786,104,A measurement of the binding microbial-induced...,Microbial-induced IgM Antibody Measurement
4,C189493,73,A score system used to evaluate the severity o...,RAST Score
5,C25613,65,One hundred times the quotient of one quantity...,Percentage
6,C105706,51,The non-cellular portion of the circulating bl...,Serum or Plasma
7,C12434,44,A liquid tissue with the primary function of t...,Blood
8,C13283,40,The fluid produced by the kidneys.,Urine
9,C13325,39,The clear portion of the blood that remains af...,Serum


## 8. Build `Unresolved` sheet — gaps for human review

Any `codelist_concept_id` or `assigned_term_concept_id` that the SDTM CT package does not know about. Does not block downstream use — just flagged.


In [8]:
ct_known_ids = set(ct['concept_id'].dropna().unique())

unresolved_codelists = [
    {'source': 'Codelists', 'concept_id': c, 'context': 'codelist_concept_id'}
    for c in sorted(referenced_codelist_ids - ct_known_ids)
]
unresolved_terms = [
    {'source': 'Variables', 'concept_id': c, 'context': 'assigned_term_concept_id'}
    for c in sorted(referenced_term_ids - ct_known_ids)
]
unresolved_sheet = pd.DataFrame(unresolved_codelists + unresolved_terms)

# Annotate usage so a reviewer can see impact
use_map_cl = dict(zip(codelists_in['codelist_concept_id'], codelists_in['variable_uses_count']))
use_map_tm = dict(zip(term_usage['assigned_term_concept_id'], term_usage['used_in_variables_count']))
def _uses(row):
    if row['source'] == 'Codelists':
        return use_map_cl.get(row['concept_id'])
    return use_map_tm.get(row['concept_id'])
if len(unresolved_sheet):
    unresolved_sheet['variable_uses_count'] = unresolved_sheet.apply(_uses, axis=1)

print(f'Unresolved sheet: {unresolved_sheet.shape}')
if len(unresolved_sheet):
    print(unresolved_sheet.to_string())


Unresolved sheet: (4, 4)
      source concept_id                   context  variable_uses_count
0  Codelists     C66790       codelist_concept_id                    1
1  Codelists     C74457       codelist_concept_id                    1
2  Variables    C132388  assigned_term_concept_id                    2
3  Variables    C171439  assigned_term_concept_id                    2


## 9. Build `Anomalies` sheet — pinned-term-not-in-bound-codelist

Some variables are pinned to a term that is NOT a permissible value of the codelist the variable is bound to. This is a data anomaly in the source — flagged here, not fixed.

Scope: only variables where both `codelist_concept_id` and `assigned_term_concept_id` are populated AND both resolve against the CT package. Rows where one or both IDs are unresolved are already in the `Unresolved` sheet.


In [9]:
# Set of valid (codelist, term) memberships from CT
valid_pairs = set(zip(
    codelist_terms_sheet['codelist_concept_id'],
    codelist_terms_sheet['term_concept_id'],
))

# Variables rows with both IDs populated and both known to CT
vpair = variables_in[
    variables_in['codelist_concept_id'].notna()
    & variables_in['assigned_term_concept_id'].notna()
    & variables_in['codelist_concept_id'].isin(ct_known_ids)
    & variables_in['assigned_term_concept_id'].isin(ct_known_ids)
].copy()

vpair['is_member'] = [
    (c, t) in valid_pairs
    for c, t in zip(vpair['codelist_concept_id'], vpair['assigned_term_concept_id'])
]

anomalies_sheet = vpair.loc[~vpair['is_member'], [
    'ds_id', 'variable_name',
    'codelist_concept_id', 'codelist_submission_value',
    'assigned_term_concept_id', 'assigned_term_value',
]].copy()
anomalies_sheet.insert(0, 'issue_type', 'pinned_term_not_in_bound_codelist')
anomalies_sheet = anomalies_sheet.reset_index(drop=True)

print(f'Anomalies sheet: {anomalies_sheet.shape}')
if len(anomalies_sheet):
    print(anomalies_sheet.to_string())


Anomalies sheet: (0, 7)


## 10. Resolution summary

Coverage stats surfaced in the ReadMe.


In [10]:
summary = {
    'codelists_referenced':        len(referenced_codelist_ids),
    'codelists_resolved':           int(codelists_sheet['codelist_submission_value_ct'].notna().sum()),
    'codelists_unresolved':         len(referenced_codelist_ids - ct_known_ids),
    'assigned_terms_distinct':      len(referenced_term_ids),
    'assigned_terms_resolved':      int(assigned_terms_sheet['term_definition'].notna().sum()),
    'assigned_terms_unresolved':    len(referenced_term_ids - ct_known_ids),
    'codelist_terms_total_rows':    len(codelist_terms_sheet),
    'submission_value_mismatches':  int(mismatch.sum()),
    'pinned_term_not_in_codelist':  len(anomalies_sheet),
}
for k, v in summary.items():
    print(f'  {k:32s} : {v:>6,}')


  codelists_referenced             :    297
  codelists_resolved               :    295
  codelists_unresolved             :      2
  assigned_terms_distinct          :  1,307
  assigned_terms_resolved          :  1,305
  assigned_terms_unresolved        :      2
  codelist_terms_total_rows        : 17,607
  submission_value_mismatches      :      0
  pinned_term_not_in_codelist      :      0


## 11. Write `interim/COSMoS_Graph_CT.xlsx`


In [11]:
readme_lines = [
    'COSMoS Graph — CT-resolved view',
    '',
    f'Generated : {date.today().isoformat()}',
    'Pipeline  : cosmos-graph/notebooks/20_resolve_ct.ipynb',
    f'Input     : {GRAPH_XLSX.name}, {CT_TXT.name}',
    '',
    'SHEETS',
    f'  Codelists       {len(codelists_sheet):>6,} rows — input bindings + CT enrichment',
    f'  CodelistTerms   {len(codelist_terms_sheet):>6,} rows — one per (codelist, term) for referenced codelists',
    f'  AssignedTerms   {len(assigned_terms_sheet):>6,} rows — resolved identity for pinned terms (stable fields only)',
    f'  Unresolved      {len(unresolved_sheet):>6,} rows — concept IDs not in SDTM_Terminology.txt',
    f'  Anomalies       {len(anomalies_sheet):>6,} rows — pinned term not member of bound codelist',
    '',
    'COVERAGE',
]
for k, v in summary.items():
    readme_lines.append(f'  {k:32s} : {v:,}')
readme_lines += [
    '',
    'JOIN GUIDE',
    '  Variables.codelist_concept_id       -> Codelists.codelist_concept_id         (codelist identity)',
    '  Variables.codelist_concept_id       -> CodelistTerms.codelist_concept_id     (full permissible value list)',
    '  Variables.assigned_term_concept_id  -> AssignedTerms.assigned_term_concept_id (stable term identity)',
    '  Variables.(codelist_concept_id, assigned_term_concept_id)',
    '    -> CodelistTerms.(codelist_concept_id, term_concept_id)   (context-specific submission_value, synonyms)',
    '',
    'See cosmos-graph/docs/COSMoS_Graph.md for the graph model and design rationale.',
    'CT source: NCI EVS SDTM Terminology package 2026-03-27.',
]

readme_df = pd.DataFrame({'README': readme_lines})

with pd.ExcelWriter(OUT_XLSX, engine='openpyxl') as writer:
    readme_df.to_excel(writer, sheet_name='ReadMe', index=False, header=False)
    codelists_sheet.to_excel(writer, sheet_name='Codelists', index=False)
    codelist_terms_sheet.to_excel(writer, sheet_name='CodelistTerms', index=False)
    assigned_terms_sheet.to_excel(writer, sheet_name='AssignedTerms', index=False)
    unresolved_sheet.to_excel(writer, sheet_name='Unresolved', index=False)
    anomalies_sheet.to_excel(writer, sheet_name='Anomalies', index=False)

print(f'wrote {OUT_XLSX}')
print(f'size : {OUT_XLSX.stat().st_size:,} bytes')


wrote /home/claude/work/cdisc-for-ai/cosmos-graph/interim/COSMoS_Graph_CT.xlsx
size : 1,386,661 bytes


## 12. Apply yellow layout

Re-open the written xlsx and apply the shared cosmos-track layout pattern: gold header, pale yellow data fill, thin borders, wrap-text cells, auto-widths. Same helper as notebook 10.


In [12]:
from openpyxl import load_workbook
from openpyxl.styles import PatternFill, Font, Border, Side, Alignment

YELLOW_HEADER = PatternFill(start_color='FFD700', end_color='FFD700', fill_type='solid')
YELLOW_FILL   = PatternFill(start_color='FFFCE8', end_color='FFFCE8', fill_type='solid')
README_HEADER = PatternFill(start_color='595959', end_color='595959', fill_type='solid')
HEADER_FONT   = Font(bold=True)
HEADER_WHITE  = Font(bold=True, color='FFFFFF')
thin_border = Border(
    left=Side(style='thin', color='999999'),
    right=Side(style='thin', color='999999'),
    top=Side(style='thin', color='999999'),
    bottom=Side(style='thin', color='999999'),
)

def auto_width(ws, max_width=50):
    for col in ws.columns:
        width = max((len(str(c.value)) for c in col if c.value is not None), default=10)
        ws.column_dimensions[col[0].column_letter].width = min(width + 2, max_width)

def format_data_sheet(ws):
    for cell in ws[1]:
        cell.font = HEADER_FONT
        cell.fill = YELLOW_HEADER
        cell.border = thin_border
        cell.alignment = Alignment(wrap_text=True, vertical='top')
    for row in ws.iter_rows(min_row=2, max_row=ws.max_row):
        for cell in row:
            cell.fill = YELLOW_FILL
            cell.border = thin_border
            cell.alignment = Alignment(wrap_text=True, vertical='top')
    auto_width(ws)

wb = load_workbook(OUT_XLSX)
for sheet_name in ('Codelists', 'CodelistTerms', 'AssignedTerms', 'Unresolved', 'Anomalies'):
    format_data_sheet(wb[sheet_name])

# ReadMe — single-column, dark grey header band on row 1
ws = wb['ReadMe']
ws['A1'].font = HEADER_WHITE
ws['A1'].fill = README_HEADER
ws['A1'].alignment = Alignment(vertical='top')
ws.column_dimensions['A'].width = 80
wb.save(OUT_XLSX)
print(f're-saved with yellow layout: {OUT_XLSX}')


re-saved with yellow layout: /home/claude/work/cdisc-for-ai/cosmos-graph/interim/COSMoS_Graph_CT.xlsx


## 13. Summary


In [13]:
xl = pd.ExcelFile(OUT_XLSX)
print(f'sheets in output: {xl.sheet_names}')
for s in xl.sheet_names:
    sh = pd.read_excel(OUT_XLSX, sheet_name=s, header=None if s == 'ReadMe' else 0)
    print(f'  {s:15s}: {sh.shape}')


sheets in output: ['ReadMe', 'Codelists', 'CodelistTerms', 'AssignedTerms', 'Unresolved', 'Anomalies']
  ReadMe         : (33, 1)
  Codelists      : (297, 7)


  CodelistTerms  : (17607, 7)
  AssignedTerms  : (1307, 4)
  Unresolved     : (4, 4)
  Anomalies      : (0, 7)
